In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import ssl
import pandas as pd
from urllib.error import URLError
from urllib.request import urlopen

In [7]:
DATA_URL = "https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv"

try:
    df = pd.read_csv(DATA_URL, sep=";")
except URLError:
    ssl_context = ssl._create_unverified_context()
    with urlopen(DATA_URL, context=ssl_context) as response:
        df = pd.read_csv(response, sep=";")

df["Income"] = df["Income"].fillna(round(df["Income"].mean(), 1))

In [9]:
df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


##Feature prep

In [11]:
df["Age"] = 2014 - df["Year_Birth"]
df["Total_Spending"] = df[["MntWines", "MntFruits", "MntMeatProducts",
 "MntFishProducts", "MntSweetProducts", "MntGoldProds"]].sum(axis=1)
df["Total_Purchases"] = df[["NumDealsPurchases", "NumWebPurchases",
"NumCatalogPurchases", "NumStorePurchases"]].sum(axis=1)
df["Children"] = df["Kidhome"] + df["Teenhome"]

features = ["Income", "Age", "Total_Spending", "Total_Purchases", "Recency", "Children"]
X = df[features]

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [16]:
sil_scores = {}
for k in range(2, 10):
 hc = AgglomerativeClustering(n_clusters=k, linkage="ward")
labels = hc.fit_predict(X_scaled)
sil_scores[k] = silhouette_score(X_scaled, labels)

print("Silhouette scores by k:", {k: round(v, 3) for k, v in sil_scores.items()})
best_k = max(sil_scores, key=sil_scores.get)
print(f"Best k: {best_k}")

Silhouette scores by k: {9: 0.17}
Best k: 9


In [18]:
hc_final = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
df["HC_Cluster"] = hc_final.fit_predict(X_scaled)
print(f"Final silhouette score: {silhouette_score(X_scaled, df['HC_Cluster']):.3f}")
print(df["HC_Cluster"].value_counts().sort_index())

Final silhouette score: 0.170
HC_Cluster
0    369
1    312
2    252
3    229
4    216
5      1
6    239
7    325
8    297
Name: count, dtype: int64


In [19]:
linkage_matrix = linkage(X_scaled, method="ward")

In [20]:
df.to_csv("hierarchical_clustered_data.csv", index=False)